# Final Paper Plots

In [1]:
import glob
import matplotlib.pyplot as plt
import sys
import xarray as xr
import numpy as np
import pandas as pd
import re
from matplotlib.lines import Line2D
import os
from matplotlib import cm
import matplotlib.patches as mpatches
from matplotlib import ticker

sys.path.append('/home/chinahg/GCresearch/contrailuncertainty/start_here/')
import pipeline_fxn_lib as lib
sys.path.append('/home/chinahg/GCresearch/contrailuncertainty/LRT/')
import LRT_fxnlib as LRTlib

plt.rcParams["font.family"] = "FreeSerif"
plt.rcParams["mathtext.fontset"] = "cm"

In [ ]:
# Import APCEMM, CoCiP, and LES bypass data
test_ids = ['110T205L25', '110T218L25', '110T225L25', '130T205L25', '130T218L25', '130T225L25']

base_dir = '/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES'

# APCEMM
apcemm_bypass_data = {
    tid: lib.read_apcemm_data(f'{base_dir}/APCEMM/1000-bins-optimized/{tid}/outputs').ds_t
    for tid in test_ids
}

# CoCiP
cocip_bypass_data = {
    tid: xr.open_dataset(f'{base_dir}/CoCiP/{tid}/{tid}-bypass_midnight.nc')
    for tid in test_ids
}

# LES
LES_data = {
    tid: pd.read_csv(f'{base_dir}/LES/processed_data/all_data/{tid}.csv')
    for tid in test_ids
}


In [ ]:
cocip_met = xr.open_dataset("/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/CoCiP/110T218L25/110T218L25-met-midnight.nc", decode_times=False)
apcemm_met = xr.open_dataset("/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/APCEMM/base_inputs/110T218L25/110T218L25.nc", decode_times=False)

In [ ]:
cocip_met.sel(time=0)["eastward_wind"][0][0].values

In [ ]:
apcemm_met.sel(time=0)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), dpi=150)
altitude_a = apcemm_met.sel(time=0)["altitude"].values
altitude_c = cocip_met.sel(time=0)["altitude"].values

shear_a = apcemm_met.sel(time=0)["shear"].values
shear_c = np.abs(np.diff(cocip_met.sel(time=0)["eastward_wind"][0][0].values)/np.diff(cocip_met.sel(time=0)["altitude"].values))
print(shear_c)

ax.plot(shear_a, altitude_a, label="APCEMM", color="blue")
ax.plot(shear_c, altitude_c[:-1], label="CoCiP", color="orange")
ax.set_ylim(10000, 12000)

# Figure X: Contrail Representation in LRT

In [ ]:
# load CSV and plot Net tau vs z [km]
path = '/home/chinahg/GCresearch/contrailuncertainty/LRT/vertical_tau_example_profile.csv'
df = pd.read_csv(path)

tau_col = df.columns[4]
z_col = df.columns[0]

z = df[z_col].astype(float).values
tau = df[tau_col].astype(float).values

plt.figure(figsize=(2,3), dpi=600)
plt.plot(tau, z, color='black', linewidth=1)
plt.xlabel(r'$\tau$')
plt.ylabel('z [km]')
plt.xscale('log')

ax = plt.gca()
ax.xaxis.set_major_locator(ticker.LogLocator(base=10.0, numticks=6))
ax.xaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs=np.arange(1,8), numticks=6))
ax.xaxis.set_major_formatter(ticker.LogFormatterSciNotation())
ax.set_ylim(0, 65)
ax.axhline(y=9, color='red', linestyle='-', linewidth=0.5)
ax.axhline(y=13, color='black', linestyle='--', linewidth=0.5)
ax.axhline(y=25, color='black', linestyle='--', linewidth=0.5)
ax.axhline(y=50, color='black', linestyle='--', linewidth=0.5)

plt.title('Vertical Optical Depth')
plt.show()

# Figure 1: RF evolution

In [ ]:
# Import APCEMM dataset
test_id = "110T218L25"
APCEMM_data = lib.read_apcemm_data(
    f'/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/APCEMM/1000-bins-optimized/{test_id}/outputs'
)

APCEMM_ds = APCEMM_data.ds_t

In [ ]:
APCEMM_ds[0]

In [ ]:
timesteps_hours = np.arange(len(APCEMM_ds)) * 10 / 60  # Convert each timestep to hours
timesteps = [0,1,2,3,4,5] #np.arange(len(APCEMM_ds))
num_timesteps = len(timesteps)
max_num_slices_averaged = 25

APCEMM_depth = []
APCEMM_IWC_avg = []
APCEMM_radii = []

# variable-length per timestep (store each timestep's slice array as an object)
APCEMM_IWC_slice_avg = np.empty(len(APCEMM_ds), dtype=object)
APCEMM_radii_slice_avg = np.empty(len(APCEMM_ds), dtype=object)
APCEMM_depth_slice_avg = np.empty(len(APCEMM_ds), dtype=object)
APCEMM_ds_masked = np.empty(num_timesteps, dtype=object)  # To store masked datasets for each timestep

num_slices_array = np.zeros(num_timesteps, dtype=int)

for time_idx in timesteps:
    print(f"Processing time step {time_idx} / {num_timesteps-1} (t = {timesteps_hours[time_idx]:.2f} hours)")
    (iwc_slice_avg,
     radii_slice_avg,
     depth_slice_avg,
     iwc,
     radii,
     depth) = LRTlib.prepare_APCEMM_slices(
        APCEMM_ds, time_idx, max_num_slices_averaged
    )

    # fixed-length outputs
    APCEMM_IWC_slice_avg[time_idx] = iwc_slice_avg
    APCEMM_radii_slice_avg[time_idx] = radii_slice_avg
    APCEMM_depth_slice_avg[time_idx] = depth_slice_avg

    # variable-length outputs
    APCEMM_IWC_avg.append(iwc)
    APCEMM_radii.append(radii)
    APCEMM_depth.append(depth)

    num_slices_array[time_idx] = len(iwc)

    APCEMM_ds_masked[time_idx] = LRTlib.mask_IWC_95_percentile_IWC(APCEMM_ds, time_idx)


In [ ]:
timestep = 5
ds_frame = APCEMM_ds_masked[timestep]

x_vals = ds_frame["x"].values
y_vals = ds_frame["y"].values
iwc_plot = ds_frame["IWC"].values * 1e3  # g/m^3

data_radii = APCEMM_radii_slice_avg[timestep] * 1e-6
data_depth = APCEMM_depth_slice_avg[timestep]
num_to_display = len(data_radii)

valid_mask = np.isfinite(iwc_plot) & (iwc_plot != 0)
valid_rows = np.where(np.any(valid_mask, axis=1))[0]
valid_cols = np.where(np.any(valid_mask, axis=0))[0]

x_min_valid = x_vals[valid_cols].min()
x_max_valid = x_vals[valid_cols].max()
y_min_valid = y_vals[valid_rows].min()
y_max_valid = y_vals[valid_rows].max()

x_grid_width = x_vals[1] - x_vals[0]
y_grid_width = y_vals[1] - y_vals[0]

shift_x = -x_vals[0]  # Shift x values so that the minimum valid x starts at 0
x_vals_slices = np.linspace(x_min_valid, x_max_valid, num_to_display)
x_edges = np.linspace(x_min_valid-x_grid_width/2, x_max_valid+x_grid_width/2, num_to_display + 1)

fig, axes = plt.subplots(
    3, 2, figsize=(6, 7), dpi=300,
    gridspec_kw={"width_ratios": [100, 3]},
    sharex="col"
)
# axes[0, 0].set_title(f"110% RH$_i$, 218K Age {timestep*10} minutes")

axes[1, 1].remove()
axes[2, 1].remove()

im = axes[0, 0].pcolormesh(x_vals + shift_x, y_vals/1000 + 10.7, iwc_plot, shading="auto", cmap="viridis")
cbar = fig.colorbar(im, cax=axes[0, 1], label="IWC [g/m³]")
cbar.formatter = ticker.ScalarFormatter(useMathText=True)
cbar.formatter.set_powerlimits((0, 0))
cbar.update_ticks()

axes[0, 0].set_ylabel("y [km]", fontsize=16)
axes[0, 0].set_ylim(10.35, 10.8)
axes[0, 0].set_xlim(x_min_valid + shift_x - 100, x_max_valid + shift_x + 100)

valid_d = np.isfinite(data_depth)
axes[1, 0].hist(x_vals_slices[valid_d]+shift_x, bins=x_edges + shift_x, weights=data_depth[valid_d], color="tab:blue", edgecolor="black", linewidth=0.5)
axes[1, 0].set_ylabel("Depth [m]", fontsize=16)

valid_r = np.isfinite(data_radii)
axes[2, 0].hist(x_vals_slices[valid_r]+shift_x, bins=x_edges + shift_x, weights=data_radii[valid_r] * 1e6, color="tab:blue", edgecolor="black", linewidth=0.5)
axes[2, 0].set_ylabel("Effective Radius [µm]", fontsize=16)
axes[2, 0].set_xlabel("x [m]", fontsize=16)

# Add vertical bin-edge lines to all plots
for ax in [axes[0, 0], axes[1, 0], axes[2, 0]]:
    for x_edge in x_edges:
        ax.axvline(x_edge+ shift_x, color="lightgray", linewidth=0.6, alpha=0.8, zorder=0)

plt.tight_layout()
plt.show()

In [ ]:
rf_dir = f"/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/APCEMM/RF_EF_results/slicing_results/{test_id}/RF/outputs/timestep_{timestep}"
csv_files = sorted(glob.glob(os.path.join(rf_dir, "*.csv")))
df_list = [pd.read_csv(f) for f in csv_files]

# Find 0h LW file and 12h LW + SW files
lw_0h_df = None
lw_12h_df = None
sw_12h_df = None

for i, f in enumerate(csv_files):
    if "0h" in f and "LW" in f:
        lw_0h_df = df_list[i]
    elif "12h" in f and "LW" in f:
        lw_12h_df = df_list[i]
    elif "12h" in f and "SW" in f:
        sw_12h_df = df_list[i]

# Stacked vertically + histogram style (like previous cell)
fig, axes = plt.subplots(2, 1, figsize=(6, 6), dpi=300, sharex=True)
fig.suptitle(f"110% RH$_i$, 218K Age {timestep*10} minutes", y=0.98)

# 0h (LW only)
if lw_0h_df is not None and "LW_Radiative_Forcing_W_m2" in lw_0h_df:
    y0 = lw_0h_df["LW_Radiative_Forcing_W_m2"].to_numpy(dtype=float)
    m0 = np.isfinite(y0)
    axes[0].hist(
        x_vals_slices[:-1][m0], bins=x_edges[1:-1][m0], weights=y0[m0],
        color="tab:red", edgecolor="black", linewidth=0.5
    )
axes[0].set_title("LW Only")
axes[0].set_ylabel("RF [W/m²]")

# 12h (LW + SW)
if (
    lw_12h_df is not None and sw_12h_df is not None
    and "LW_Radiative_Forcing_W_m2" in lw_12h_df
    and "SW_Radiative_Forcing_W_m2" in sw_12h_df
):
    y12 = (
        lw_12h_df["LW_Radiative_Forcing_W_m2"].to_numpy(dtype=float)
        + sw_12h_df["SW_Radiative_Forcing_W_m2"].to_numpy(dtype=float)
    )
    m12 = np.isfinite(y12)
    axes[1].hist(
        x_vals_slices[:-1][m12], bins=x_edges[1:-1][m12], weights=y12[m12],
        color="tab:orange", edgecolor="black", linewidth=0.5
    )
axes[1].set_title("LW + SW")
axes[1].set_ylabel("RF [W/m²]")
axes[1].set_xlabel("x [m]")

# Add vertical bin-edge lines to both subplots (same style as previous cell)
for ax in axes:
    for x_edge in x_edges:
        ax.axvline(x_edge, color="lightgray", linewidth=0.6, alpha=0.8, zorder=0)

plt.tight_layout()
plt.show()

# Figure 2: Contrail Properties

In [ ]:
temp_groups = ['205', '218', '225']
color_map = {
    '205': 'gray',
    '218': 'red',
    '225': 'green'
}

In [ ]:
def get_apcemm_sauter_reff_um(ds):
    n = np.asarray(ds["Overall size distribution"]).squeeze().astype(float)
    r_e = np.asarray(ds["r_e"]).squeeze().astype(float)

    if r_e.size == n.size + 1:
        r_mid = 0.5 * (r_e[:-1] + r_e[1:])
    else:
        r_mid = r_e[:n.size]

    den = np.sum(n * r_mid**2)
    if den <= 0:
        return np.nan
    return (np.sum(n * r_mid**3) / den) * 1e6  # um

In [ ]:
rho_ice = 917.0  # kg/m^3, used for CoCiP mass calculation
model_colors = {"APCEMM": "tab:blue", "LES": "tab:orange", "CoCiP": "tab:green"}
rh_styles = {110: "--", 130: "-"}

In [ ]:
# 3 rows (reff, number, mass) x 3 cols (205K, 218K, 225K)
fig, axes = plt.subplots(
    3, 3, figsize=(14, 6), dpi=200,
    sharex='col', sharey='row', layout='constrained'
)

for j, temp in enumerate(temp_groups):
    ax_reff = axes[0, j]
    ax_num = axes[1, j]
    ax_mass = axes[2, j]

    ax_num.set_yscale("log")
    ax_mass.set_yscale("linear")
    ax_reff.set_title(f"{temp}K")

    max_age = 0.0  # track max plotted age for this temperature column

    for rh in [110, 130]:
        tid = f"{rh}T{temp}L25"
        ls = rh_styles[rh]

        # APCEMM
        if tid in apcemm_bypass_data:
            ds_list = apcemm_bypass_data[tid]
            t_ap = np.arange(len(ds_list)) * 10 / 60
            r_ap_um = np.array([get_apcemm_sauter_reff_um(ds) for ds in ds_list], dtype=float)
            n_ap = np.array([ds["Number Ice Particles"].item() for ds in ds_list], dtype=float)
            m_ap = np.array([ds["Ice Mass"].item() for ds in ds_list], dtype=float)

            ax_reff.plot(t_ap, r_ap_um, color=model_colors["APCEMM"], linestyle=ls, alpha=0.95)
            ax_num.plot(t_ap, n_ap, color=model_colors["APCEMM"], linestyle=ls, alpha=0.95)
            ax_mass.plot(t_ap, m_ap, color=model_colors["APCEMM"], linestyle=ls, alpha=0.95)

            if t_ap.size:
                max_age = max(max_age, float(np.nanmax(t_ap)))

        # LES
        if tid in LES_data:
            t_les = LES_data[tid]["Time_hours"]

            if "Effective_radius_um" in LES_data[tid]:
                ax_reff.plot(
                    t_les,
                    LES_data[tid]["Effective_radius_um"],
                    color=model_colors["LES"], linestyle=ls, alpha=0.9
                )
                print(f"{tid} LES initial r_eff: {LES_data[tid]['Effective_radius_um'].values[0]:.2f} um")
            ax_num.plot(
                t_les,
                LES_data[tid]["Ice_number"],
                color=model_colors["LES"], linestyle=ls, alpha=0.9
            )
            ax_mass.plot(
                t_les,
                LES_data[tid]["Ice_mass"],
                color=model_colors["LES"], linestyle=ls, alpha=0.9
            )

            t_les_arr = np.asarray(t_les, dtype=float)
            if t_les_arr.size:
                max_age = max(max_age, float(np.nanmax(t_les_arr)))

        # CoCiP
        if tid in cocip_bypass_data:
            cds = cocip_bypass_data[tid]
            t_co = cds["age_hours"].values
            r_co_um = cds["r_ice_vol"].values * 1e6
            n_co = cds["n_ice_per_m"].values
            m_co = (4.0 / 3.0) * np.pi * rho_ice * (cds["r_ice_vol"].values**3) * n_co

            ax_reff.plot(t_co, r_co_um, color=model_colors["CoCiP"], linestyle=ls, alpha=0.9)
            ax_num.plot(t_co, n_co, color=model_colors["CoCiP"], linestyle=ls, alpha=0.9)
            ax_mass.plot(t_co, m_co, color=model_colors["CoCiP"], linestyle=ls, alpha=0.9)

            if t_co.size:
                max_age = max(max_age, float(np.nanmax(t_co)))

    # Set x-limits/ticks for this temperature column: ceil to nearest 5 h
    x_max = int(5 * np.ceil(max_age / 5.0)) if max_age > 0 else 5
    for ax in axes[:, j]:
        ax.set_xlim(0, x_max)
        ax.set_xticks(np.arange(0, x_max + 1, 5))
        ax.minorticks_off()

# Axis labels
axes[0, 0].set_ylabel("Sauter Effective Radius\n[$\\mu m$]")
axes[1, 0].set_ylabel("Crystal Number \n[$\\#/m$]")
axes[2, 0].set_ylabel("Ice Mass \n[$kg/m$]")
for j in range(3):
    axes[2, j].set_xlabel("Contrail Age [hours]")

# Legends: model=color, RH=linestyle
model_handles = [
    Line2D([0], [0], color=model_colors["APCEMM"], lw=2, linestyle="-", label="APCEMM"),
    Line2D([0], [0], color=model_colors["LES"], lw=2, linestyle="-", label="LES"),
    Line2D([0], [0], color=model_colors["CoCiP"], lw=2, linestyle="-", label="CoCiP"),
]
rh_handles = [
    Line2D([0], [0], color="k", lw=2, linestyle=rh_styles[110], label=r"$RH_i=110\%$"),
    Line2D([0], [0], color="k", lw=2, linestyle=rh_styles[130], label=r"$RH_i=130\%$"),
]

fig.legend(
    handles=model_handles + rh_handles,
    loc="lower center", bbox_to_anchor=(0.5, -0.075),
    ncol=5, frameon=False
)

plt.show()

In [ ]:
tid = "110T225L25"
print(float(cocip_bypass_data[tid]["r_ice_vol"].values[0]))

In [ ]:
# Print initial (first valid) effective radius, ice number, and ice mass
# for LES, APCEMM, and CoCiP, organized by temperature and RH_i.

def _first_valid(arr):
    a = np.asarray(arr, dtype=float).ravel()
    a = a[np.isfinite(a)]
    return float(a[0]) if a.size else np.nan

def _fmt_val(v, unit="", sci=False):
    if not np.isfinite(v):
        return "nan"
    return f"{v:.3e} {unit}".strip() if sci else f"{v:.3f} {unit}".strip()

# Sort cases by temperature, then RH (e.g., 205K: 110%, 130%)
case_info = []
for tid in test_ids:
    m = re.match(r"(\d+)T(\d+)L\d+", tid)
    if m:
        rh_i = int(m.group(1))
        temp_k = int(m.group(2))
        case_info.append((temp_k, rh_i, tid))
case_info = sorted(case_info, key=lambda x: (x[0], x[1]))

for temp_k, rh_i, tid in case_info:
    # LES
    les_reff = np.nan
    les_n = np.nan
    les_m = np.nan
    if tid in LES_data:
        dfl = LES_data[tid]
        if "Effective_radius_um" in dfl.columns:
            les_reff = _first_valid(dfl["Effective_radius_um"].values)
        if "Ice_number" in dfl.columns:
            les_n = _first_valid(dfl["Ice_number"].values)
        if "Ice_mass" in dfl.columns:
            les_m = _first_valid(dfl["Ice_mass"].values)

    # APCEMM
    ap_reff = np.nan
    ap_n = np.nan
    ap_m = np.nan
    if tid in apcemm_bypass_data and len(apcemm_bypass_data[tid]) > 0:
        ds0 = apcemm_bypass_data[tid][0]
        ap_reff = get_apcemm_sauter_reff_um(ds0)
        ap_n = float(ds0["Number Ice Particles"].item()) if "Number Ice Particles" in ds0 else np.nan
        ap_m = float(ds0["Ice Mass"].item()) if "Ice Mass" in ds0 else np.nan

    # CoCiP
    co_reff = np.nan
    co_n = np.nan
    co_m = np.nan
    if tid in cocip_bypass_data:
        cds = cocip_bypass_data[tid]
        co_reff = _first_valid(cds["r_ice_vol"].values * 1e6)  # um
        co_n = _first_valid(cds["n_ice_per_m"].values)         # #/m
        if np.isfinite(co_reff) and np.isfinite(co_n):
            co_m = (4.0 / 3.0) * np.pi * rho_ice * (co_reff * 1e-6) ** 3 * co_n  # kg/m

    print(f"{temp_k}K {rh_i}%:")
    print(f"  LES:    initial Effective Radius={_fmt_val(les_reff, 'micron')}, Ice Number={_fmt_val(les_n, '#/m', sci=True)}, Ice Mass={_fmt_val(les_m, 'kg/m', sci=True)}")
    print(f"  APCEMM: initial Effective Radius={_fmt_val(ap_reff, 'micron')}, Ice Number={_fmt_val(ap_n, '#/m', sci=True)}, Ice Mass={_fmt_val(ap_m, 'kg/m', sci=True)}")
    print(f"  CoCiP:  initial Effective Radius={_fmt_val(co_reff, 'micron')}, Ice Number={_fmt_val(co_n, '#/m', sci=True)}, Ice Mass={_fmt_val(co_m, 'kg/m', sci=True)}")
    print()

# Figure: 1000 bins vs 2 bins vs CoCiP

In [ ]:
ds_cocip['sin_a'].values

In [ ]:
sigma_yz = ds_cocip["sigma_yz"].values
times = ds_cocip["age_hours"].values

fig, ax = plt.subplots(figsize=(6, 4), dpi=150)
ax.plot(times, sigma_yz)
ax.set_xlabel("Contrail Age [hours]")
ax.set_ylabel(r"$\sigma_{yz}$ [m²/s²]")

In [ ]:
import matplotlib.colors as mcolors
# ── Configuration ────────────────────────────────────────────────────────────

TEST_ID    = '110T218L25'
GRID_RES   = 220
X_BUFFER   = 0.10
KEY_A = "Effective radius"
KEY_C = "r_ice_vol"
KEY_MASS = "Effective radius"  # mass-weighting field; set to None to use uniform weights
MASK_THRESHOLD = 5e-8          # values below this are masked in APCEMM rows

# ── Load datasets ─────────────────────────────────────────────────────────────

ds_cocip     = xr.open_dataset(f"/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/CoCiP/{TEST_ID}/{TEST_ID}-bypass_midnight.nc")
ds_1000_bins = lib.read_apcemm_data(f"/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/APCEMM/1000-bins-optimized/{TEST_ID}/outputs/").ds_t
ds_2_bins    = lib.read_apcemm_data(f"/home/chinahg/GCresearch/contrailuncertainty/APCEMM_vs_CoCiP_vs_LES/APCEMM/testing/2-bins-TP6-TT1/{TEST_ID}/outputs/").ds_t

# ── Helpers ───────────────────────────────────────────────────────────────────

def mass_weighted_centroid(ds, key_mass=None, threshold=None):
    """
    Returns (x_c, y_c) mass-weighted centroid.
    If key_mass is None or not present in ds, falls back to uniform weighting.
    Cells where the weight field is below threshold are excluded.
    """
    x_vals = ds["x"].values
    y_vals = ds["y"].values

    if key_mass is not None and key_mass in ds:
        weights = np.where(np.isnan(ds[key_mass].values), 0.0, ds[key_mass].values)
    else:
        weights = np.ones((len(y_vals), len(x_vals)))

    if threshold is not None:
        weights = np.where(weights < threshold, 0.0, weights)

    X, Y = np.meshgrid(x_vals, y_vals)

    total = weights.sum()
    if total == 0:
        x_c = x_vals.mean()
        y_c = y_vals.mean()
    else:
        x_c = (weights * X).sum() / total
        y_c = (weights * Y).sum() / total

    return x_c, y_c


def build_cocip_ellipse(time_idx):
    width         = ds_cocip["width"].values[time_idx]
    area          = ds_cocip["area_eff"].values[time_idx]
    depth         = ds_cocip["depth"].values[time_idx]
    sigma_yz      = ds_cocip["sigma_yz"].values[time_idx]
    bulk_property = ds_cocip[KEY_C].values[time_idx] * 1e6
    contrail_age_sec = ds_cocip["age_hours"].values[time_idx] * 60 * 60

    print(f"cocip width: {width:.3f} m")
    print(f"cocip depth: {depth:.3f} m")

    print(f"cocip sigma_yz: {sigma_yz:.3f}")
    print(f"Contrail age [hours]: {ds_cocip['age_hours'].values[time_idx]:.2f}")

    sigma_yy = 0.125 * width**2
    sigma_zz = 0.125 * depth**2

    # dudz = -0.004 # m/s/m

    # S = dudz * contrail_age_sec
    # print(f"Shear strain S = {S:.3f}")
    # sigma_yz = S * sigma_zz

    dudz = sigma_yz / sigma_zz / contrail_age_sec
    print(f"Back-calculated dudz = {dudz:.6f} s⁻¹")
    print(f"calculated Sigma_yz = {sigma_yz:.3f}")

    cov = np.array([[sigma_yy, sigma_yz],
                    [sigma_yz, sigma_zz]])
    
    print(f"Covariance matrix:\n{cov}")


    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    eigenvalues = np.maximum(eigenvalues, 1e-6)
    L = eigenvectors @ np.diag(np.sqrt(eigenvalues))

    t = np.linspace(0, 2 * np.pi, 500)
    circle = np.array([np.cos(t), np.sin(t)])
    sheared = L @ circle

    print(L)

    current_area = np.pi * np.sqrt(np.linalg.det(cov))
    scale = np.sqrt(area / current_area)

    x_ell = sheared[0] * scale
    y_ell = sheared[1] * scale

    print(x_ell)

    shear_angle_rad = 0.5 * np.arctan2(2 * sigma_yz, sigma_yy - sigma_zz)
    shear_angle_deg = -1 *np.degrees(shear_angle_rad)
    print(f"Shear angle (ellipse major axis relative to y): {shear_angle_deg:.2f} degrees")

    x_pad = (x_ell.max() - x_ell.min()) * X_BUFFER
    x_grid = np.linspace(x_ell.min() - x_pad, x_ell.max() + x_pad, GRID_RES)
    y_extent = np.abs(y_ell).max()

    return dict(
        x_ell=x_ell, y_ell=y_ell,
        x_grid=x_grid, y_extent=y_extent,
        bulk_value=bulk_property,
        angle_deg=shear_angle_deg,
        a_sol=None, b_sol=None,
        altitude=ds_cocip["altitude"].values[time_idx],
    )

# ── Combined plot ─────────────────────────────────────────────────────────────

plot_times_min = [30]

datasets = [ds_1000_bins, ds_2_bins]
labels   = ["1000 Bins", "2 Bins", "CoCiP"]

ncols    = len(plot_times_min)
nrows    = len(datasets) + 1
colormap = "viridis"

cbar_min, cbar_max = 1, 30
shared_norm = mcolors.Normalize(vmin=cbar_min, vmax=cbar_max)

cocip_data = {idx: build_cocip_ellipse(idx) for idx in plot_times_min}

fig, axes = plt.subplots(nrows, ncols, figsize=(8, 7), dpi=300)

cmap = plt.cm.get_cmap(colormap)

all_ymins, all_ymaxs = [], []

# ── collect CoCiP y extents up front so they feed into the shared limits ──────
for idx in plot_times_min:
    cd = cocip_data[idx]
    all_ymins.append(cd["y_ell"].min())
    all_ymaxs.append(cd["y_ell"].max())

for col, (idx, t_min) in enumerate(zip(plot_times_min, plot_times_min)):

    # — rows 0 & 1: APCEMM datasets —
    for row, ds_list in enumerate(datasets):
        ax = axes[row] if ncols == 1 else axes[row, col]

        if int(idx / 10) < len(ds_list):
            ds     = ds_list[int(idx / 10)]
            x_vals = ds["x"].values
            y_vals = ds["y"].values
            data   = ds[KEY_A].values * 1e6

            x_offset  = x_vals.min()
            x_shifted = x_vals - x_offset

            raw = ds[KEY_A].values
            masked_data = np.ma.masked_where(raw < MASK_THRESHOLD, data)

            ax.imshow(
                masked_data, cmap=cmap, norm=shared_norm,
                origin="lower", aspect="auto",
                extent=[x_shifted.min(), x_shifted.max(), y_vals.min(), y_vals.max()]
            )
            all_ymins.append(y_vals.min())
            all_ymaxs.append(y_vals.max())
            ax.axhline(y=-500, color="red", linestyle="-", linewidth=1)

            use_mass_key = KEY_MASS if row == 0 else None
            x_c, y_c = mass_weighted_centroid(ds, key_mass=use_mass_key,
                                              threshold=MASK_THRESHOLD)
            x_c_shifted = x_c - x_offset

            ax.plot(x_c_shifted, y_c, marker="x", ms=8, mew=1.5,
                    color="black", zorder=5, label="Centroid")

    # — row 2: CoCiP ellipse —
    ax_cocip = axes[-1] if ncols == 1 else axes[2, col]
    cocip_plot_ds = cocip_data[idx]

    x_ell_shifted  = cocip_plot_ds["x_ell"]
    x_grid_shifted = cocip_plot_ds["x_grid"]
    y_ell_shifted  = cocip_plot_ds["y_ell"]

    face_color = cmap(shared_norm(cocip_plot_ds["bulk_value"]))
    ax_cocip.fill(x_ell_shifted, y_ell_shifted, color=face_color, zorder=2)
    ax_cocip.plot(x_ell_shifted, y_ell_shifted, color="none", lw=0.8, zorder=3)

    ax_cocip.set_xlim(x_grid_shifted[0], x_grid_shifted[-1])
    ax_cocip.axhline(y=-500, color="red", linestyle="-", linewidth=1)

    x_c_cocip = np.mean(x_ell_shifted)
    y_c_cocip = np.mean(y_ell_shifted)
    ax_cocip.plot(x_c_cocip, y_c_cocip, marker="x", ms=8, mew=1.5,
                  color="black", zorder=5)

    # (axes[0] if ncols == 1 else axes[0, col]).set_title(
    #     f"Contrail Age {t_min} min", fontsize=10, fontweight="bold"
    # )

# — unified y-limits across ALL rows including CoCiP ─────────────────────────
y_lo, y_hi = min(all_ymins), np.abs(min(all_ymins)) #max(all_ymaxs)
for col in range(ncols):
    for row in range(nrows):
        ax = axes[row] if ncols == 1 else axes[row, col]
        ax.set_ylim(y_lo, y_hi)

# — row labels —
for row, label in enumerate(labels):
    ax = axes[row] if ncols == 1 else axes[row, 0]
    ax.set_ylabel(label, fontsize=11, fontweight="bold")

# — single shared colorbar —
fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.90, 0.05, 0.015, 0.88])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=shared_norm)
sm.set_array([])
fig.colorbar(sm, cax=cbar_ax, label="Effective radius (μm)")

# fig.suptitle("Effective radius (μm)", fontsize=13, fontweight="bold", y=0.95)
plt.tight_layout(rect=[0, 0, 0.89, 0.98])
plt.show()

In [ ]:
ds_1000_bins[0]['depth'].values

# Figure 3: Energy Forcing Effects

In [ ]:
apcemm_rf_ef = '/home/chinahg/GCresearch/contrailuncertainty/LRT/RF_EF_results/APCEMM'
cocip_rf_ef  = '/home/chinahg/GCresearch/contrailuncertainty/LRT/RF_EF_results/CoCiP'
les_rf_ef    = '/home/chinahg/GCresearch/contrailuncertainty/LRT/RF_EF_results/LES'

time_tags = ['0h', '12h']
habits = ['solid-column', 'ghm', 'rough-aggregate']
models = {'APCEMM': apcemm_rf_ef, 'CoCiP': cocip_rf_ef, 'LES': les_rf_ef}
rh_color = {'110': 'green', '130': 'orange'}

test_ids = ['110T225L25']

## Compute stats for lifetime and 4h EF

In [ ]:
data = xr.open_dataset('/home/chinahg/GCresearch/contrailuncertainty/LRT/RF_EF_results/CoCiP/110T225L25/CoCiP_110T225L25_0h_ghm.nc', decode_times=False)

In [ ]:
data["Energy Forcing"].shape

In [ ]:
# Plot Energy Forcing vs t from `data`
t_vals = data["time"].values
ef_vals = data["Energy Forcing"].values

print(f"ef_vals: {ef_vals}")

# If Energy Forcing has extra dims (e.g., slice), average over non-time axes
if ef_vals.ndim > 1:
    ef_vals = np.nanmean(ef_vals, axis=tuple(range(1, ef_vals.ndim)))

plt.figure(figsize=(6,3), dpi=200)
plt.plot(t_vals, ef_vals, marker='o', linestyle='-', color='tab:blue')
plt.xlabel('t [hours]')
plt.ylabel('Energy Forcing')
plt.title('Energy Forcing vs t')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
per_case_4h = {tid: {m: {tt: [] for tt in time_tags} for m in models} for tid in test_ids}
per_case    = {tid: {m: {tt: [] for tt in time_tags} for m in models} for tid in test_ids}

TARGET_AGE_H = 4.0

# ── helper: get the age coordinate for each model ────────────────────────────
def get_age_coord(ds, m):
    """Return age in hours as a numpy float array for the given model."""
    if m == 'APCEMM':
        return ds['t'].values.astype(float)
    elif m == 'LES':
        return ds['time'].values.astype(float)
    elif m == 'CoCiP':
        return ds['age_hours'].values.astype(float)
    else:
        raise ValueError(f"Unknown model: {m}")

# ── load per_case and per_case_4h from NetCDF ─────────────────────────────────
# cocip_bypass_data loaded here too (once per tid, using first valid habit file
# for tt='0h' since bypass variables are independent of time_tag)
cocip_bypass_data = {tid: None for tid in test_ids}

for m, base in models.items():
    print(f"Processing model: {m}")
    for tid in test_ids:
        for h in habits:
            for tt in time_tags:
                ef_path = f"{base}/{tid}/{m}_{tid}_{tt}_{h}.nc"

                if not os.path.exists(ef_path):
                    continue

                try:
                    ds = xr.open_dataset(ef_path, decode_times=False)
                except Exception as e:
                    print(f"  Could not open {ef_path}: {e}")
                    continue

                # ── select rows matching this time_tag via Hour variable ───
                hour_str = tt.replace('h', '')   # '0h'->'0', '12h'->'12'

                if m == 'APCEMM':
                    time_dim = 't'
                elif m == 'LES':
                    time_dim = 'time'
                elif m == 'CoCiP':
                    time_dim = 'index'

                age   = get_age_coord(ds, m)
                ef_vals = ds['Energy Forcing'].values.astype(float)

                print(f"  {tid} {tt} {h}: found {ef_vals.shape} EF values after masking")

                # ── per_case: last EF value (full run) ────────────────────
                # APCEMM: skip first value (index 0) to match old behaviour
                if m == 'APCEMM' and len(ef_vals) > 1:
                    ef_full = ef_vals[1:]
                else:
                    ef_full = ef_vals

                if len(ef_full) > 0:
                    per_case[tid][m][tt].append(float(ef_full[-1]))

                # ── per_case_4h: last EF value at or before 4 h ───────────
                mask_4h = age <= TARGET_AGE_H
                ef_4h   = ef_vals[mask_4h]

                if len(ef_4h) > 0:
                    per_case_4h[tid][m][tt].append(float(ef_4h[-1]))

# ── compute stats (mean/min/max) ──────────────────────────────────────────────
def compute_stats(per_case_dict):
    stats = {tid: {m: {tt: None for tt in time_tags} for m in models} for tid in test_ids}
    for tid in test_ids:
        for m in models:
            for tt in time_tags:
                vals = np.array(per_case_dict[tid][m][tt], dtype=float)
                if vals.size:
                    stats[tid][m][tt] = {
                        'mean': vals.mean(),
                        'min':  vals.min(),
                        'max':  vals.max()
                    }
    return stats

stats_case    = compute_stats(per_case)
stats_case_4h = compute_stats(per_case_4h)

# ── CoCiP internal EF extraction (replaces get_cocip_internal_point) ──────────
def get_cocip_internal_point(tid, tt, target_h):
    # no noon data
    if tt == '12h':
        return np.nan

    cbd = cocip_bypass_data.get(tid)
    if cbd is None:
        return np.nan

    age_hours    = cbd['age_hours'].values.astype(float)
    ef_vals      = cbd['ef'].values.astype(float)
    seg_lengths  = cbd['segment_length'].values.astype(float)

    max_contrail_age = age_hours[-1]

    if target_h is None:
        idx = len(age_hours) - 2   # -1 after cumsum drops first element
    elif target_h >= max_contrail_age:
        return np.nan
    else:
        idx = np.argmin(np.abs(age_hours - target_h))
        if idx == 0:
            idx = 0   # cumsum index 0 corresponds to original index 1
        else:
            idx = idx - 1

    ef_cocip_internal = np.cumsum(
        ef_vals[1:] / seg_lengths[1:]
    )  # J -> J/m cumulative

    if idx >= len(ef_cocip_internal):
        return np.nan

    print(f"tid={tid}, tt={tt}, target_h={target_h} -> idx={idx}, "
          f"age={age_hours[idx+1]:.2f} h, ef_internal={ef_cocip_internal[idx]:.2e} J/m")
    return ef_cocip_internal[idx]

# ── populate cocip_internal dict ──────────────────────────────────────────────
cocip_internal = {}
for tid in test_ids:
    for tt in time_tags:
        cocip_internal[(tid, tt, 'none')] = get_cocip_internal_point(tid, tt, None)
        cocip_internal[(tid, tt, '4h')]   = get_cocip_internal_point(tid, tt, TARGET_AGE_H)



In [ ]:
stats_case

In [ ]:

# ── layout ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=300, sharey=False)

# ── group x-axis by temperature → RHi ────────────────────────────────────────
temps_unique = sorted({int(tid.split('T')[1].split('L')[0]) for tid in test_ids})

rhi_labels  = ['110%', '130%']
rh_to_label = {v: k for k, v in rh_color.items()}

groups = []
for T in temps_unique:
    tids_at_T = sorted(
        [tid for tid in test_ids if f"T{T}" in tid],
        key=lambda s: int(s[:3])
    )
    for tid in tids_at_T:
        groups.append((T, tid))

n_groups    = len(groups)
model_order = ['CoCiP', 'LES', 'APCEMM']
n_models    = len(model_order)
bar_width   = 0.22
group_gap   = 0.12

def group_x_positions(groups, temps_unique, bar_width, group_gap):
    x0, positions, prev_T = 0.0, [], None
    for (T, tid) in groups:
        if prev_T is not None and T != prev_T:
            x0 += group_gap
        positions.append(x0)
        x0 += n_models * bar_width + 0.10
        prev_T = T
    return positions

group_starts = group_x_positions(groups, temps_unique, bar_width, group_gap)

model_colors = {'CoCiP': 'tab:green', 'LES': 'tab:orange', 'APCEMM': 'tab:blue'}
model_colors_light = {
    'CoCiP':  '#a8d5a2',
    'LES':    '#ffd699',
    'APCEMM': '#a8c4e8',
}

x_left  = -0.2
x_right = group_starts[-1] + n_models * bar_width + 0.2

# ── helper: draw all bars + errorbars onto a given axes ──────────────────────
def draw_bars(ax, tt):
    for gi, (T, tid) in enumerate(groups):
        x_group = group_starts[gi]
        for mi, m in enumerate(model_order):
            xc        = x_group + mi * bar_width
            xc_center = xc + bar_width / 2
            col       = model_colors[m]
            col_light = model_colors_light[m]

            s4 = stats_case_4h.get(tid, {}).get(m, {}).get(tt, None)
            sf = stats_case.get(tid, {}).get(m, {}).get(tt, None)

            if sf is not None:
                yf    = sf['mean']
                ax.bar(xc, abs(yf), bottom=min(yf, 0), width=bar_width,
                       align='edge', color=col_light, edgecolor=col,
                       linewidth=0.8, hatch='', zorder=2)
                ax.errorbar(xc_center, yf,
                            yerr=[[yf - sf['min']], [sf['max'] - yf]],
                            fmt='o', color=col, ecolor=col, elinewidth=0.8,
                            capsize=2, markersize=3, zorder=4)

            if s4 is not None:
                y4    = s4['mean']
                ax.bar(xc, abs(y4), bottom=min(y4, 0), width=bar_width,
                       align='edge', color='None', edgecolor=col,
                       linewidth=0.8, hatch='///', zorder=2)
                ax.errorbar(xc_center, y4,
                            yerr=[[y4 - s4['min']], [s4['max'] - y4]],
                            fmt='o', color=col, ecolor=col, elinewidth=0.8,
                            capsize=2, markersize=3, zorder=4)

# ── CoCiP EF_internal extraction ─────────────────────────────────────────────
def get_cocip_internal_point(tid, tt, target_h):
    # no noon data yet
    if tt == '12h':
        return np.nan

    max_contrail_age = cocip_bypass_data[tid]['age_hours'].values[-1]
    if target_h is None:
        idx = -1
    elif target_h >= max_contrail_age:
        return np.nan
    else:
        idx = np.argmin(np.abs(cocip_bypass_data[tid]['age_hours'].values.astype(float) - target_h))

    ef_cocip_internal = np.cumsum(
        cocip_bypass_data[tid]["ef"].values.astype(float)[1:] /
        cocip_bypass_data[tid]["segment_length"].values.astype(float)[1:]
    )  # J -> J/m cumulative
    print(f"tid={tid}, tt={tt}, target_h={target_h} -> idx={idx}, age={cocip_bypass_data[tid]['age_hours'].values[idx]:.2f} h, ef_internal={ef_cocip_internal[idx]:.2e} J/m")
    return ef_cocip_internal[idx]

# key: (tid, tt, mode) where mode in {"none", "4h"}
cocip_internal = {}
for tid in test_ids:
    for tt in time_tags:
        cocip_internal[(tid, tt, 'none')] = get_cocip_internal_point(tid, tt, None)
        cocip_internal[(tid, tt, '4h')]   = get_cocip_internal_point(tid, tt, TARGET_AGE_H)

# ── main panels ───────────────────────────────────────────────────────────────
cocip_idx = model_order.index('CoCiP')
for ax, tt, title in zip(axes, time_tags, ['LW Only', 'LW + SW']):
    draw_bars(ax, tt)

    # overlay CoCiP EF_internal as circles at CoCiP bar center
    for gi, (_, tid) in enumerate(groups):
        x = group_starts[gi] + (cocip_idx + 0.5) * bar_width

        # None/full: closed circle
        y_none = cocip_internal.get((tid, tt, 'none'), np.nan)
        if np.isfinite(y_none):
            ax.scatter(x, y_none, marker='s', s=12, linewidths=1.0,
                       facecolors='black', edgecolors='black', zorder=6)

        # 4h: open circle
        y_4h = cocip_internal.get((tid, tt, '4h'), np.nan)
        if np.isfinite(y_4h):
            ax.scatter(x, y_4h, marker='s', s=12, linewidths=1.2,
                       facecolors='none', edgecolors='black', zorder=6)

    ax.axhline(0, color='gray', linestyle='--', lw=0.8)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xticks([])
    ax.set_xlim(x_left, x_right)

# ── y limits ──────────────────────────────────────────────────────────────────
ax_ylims = {}
for ax, tt in zip(axes, time_tags):
    ymins, ymaxs = [], []
    for tid in test_ids:
        for m in model_order:
            for s in [stats_case_4h.get(tid, {}).get(m, {}).get(tt),
                      stats_case.get(tid, {}).get(m, {}).get(tt)]:
                if s:
                    ymins.append(s['min'])
                    ymaxs.append(s['max'])

        # include both CoCiP EF_internal points in limits
        for mode in ['none', '4h']:
            y_int = cocip_internal.get((tid, tt, mode), np.nan)
            if np.isfinite(y_int):
                ymins.append(y_int)
                ymaxs.append(y_int)

    ymin = min(ymins) if ymins else -1
    ymax = max(ymaxs) if ymaxs else 1

    pad = 0.06 * (ymax - ymin)
    if tt == '0h':
        ax.set_ylim(ymin, ymax + pad)
    else:
        ax.set_ylim(ymin - pad, ymax + pad)

    ax.set_ylabel('EF [J m$^{-1}$]', fontsize=12)
    ax_ylims[tt] = ax.get_ylim()

# ── x-tick labels ─────────────────────────────────────────────────────────────
for ax, tt, title in zip(axes, time_tags, ['LW Only', 'LW + SW']):
    ymin_ax, ymax_ax = ax_ylims[tt]
    ybot = ymin_ax
    for gi, (T, tid) in enumerate(groups):
        xc = group_starts[gi] + (n_models * bar_width) / 2
        rhi_str = tid[:3] + '%'
        offset = 0.02e10 if title == 'LW Only' else 0.05e9
        ax.text(xc, ybot - offset, rhi_str, ha='center', va='top', fontsize=12,
                transform=ax.transData, clip_on=False)

    prev_T, block_start_x = None, None
    def flush_temp_label(ax, T, bsx, bex, ybot, ymn, ymx):
        xc = (bsx + bex) / 2
        ax.text(xc, ybot - 0.08 * (ymx - ymn),
                f'{T} K', ha='center', va='top', fontsize=12, fontweight='bold',
                transform=ax.transData, clip_on=False)

    for gi, (T, tid) in enumerate(groups):
        if T != prev_T:
            if prev_T is not None:
                flush_temp_label(ax, prev_T, block_start_x,
                                 group_starts[gi-1] + n_models * bar_width,
                                 ybot, ymin_ax, ymax_ax)
            block_start_x = group_starts[gi]
            prev_T = T
    flush_temp_label(ax, prev_T, block_start_x,
                     group_starts[-1] + n_models * bar_width,
                     ybot, ymin_ax, ymax_ax)

# ── legend along bottom of figure ────────────────────────────────────────────
model_handles = [
    mpatches.Patch(facecolor=model_colors_light[m], edgecolor=model_colors[m],
                   linewidth=0.8, label=m)
    for m in model_order
]
full_patch  = mpatches.Patch(facecolor='lightgray', edgecolor='gray',
                             linewidth=0.8, label='full run')
trunc_patch = mpatches.Patch(facecolor='None', edgecolor='gray',
                             hatch='///', label='4h truncated')

internal_none_handle = plt.Line2D(
    [0], [0], marker='o', linestyle='None', color='black',
    markerfacecolor='black', markeredgecolor='black',
    markersize=6, label='CoCiP Internal'
)
internal_4h_handle = plt.Line2D(
    [0], [0], marker='o', linestyle='None', color='black',
    markerfacecolor='none', markeredgecolor='black',
    markersize=6, label='CoCiP Internal (4h)'
)

fig.legend(
    handles=model_handles + [full_patch, trunc_patch, internal_none_handle, internal_4h_handle],
    loc='lower center', ncol=8, fontsize=10, framealpha=0.9,
    bbox_to_anchor=(0.5, -0.06)
)

plt.tight_layout()
plt.subplots_adjust(bottom=0.14)
plt.show()

In [ ]:
# collect per-case per-model per-time final EF values
tid_colors = {tid: cm.get_cmap('tab10')(i % 10) for i, tid in enumerate(test_ids)}
per_case = {tid: {m: {tt: [] for tt in time_tags} for m in models} for tid in test_ids}

for m, base in models.items():
    apcemm_style = (m == 'APCEMM')
    for tid in test_ids:
        for h in habits:
            for tt in time_tags:
                ef_path = f"{base}/{tid}/EF/{tid}_EF_{h}_{tt}_LRT.csv"
                if os.path.exists(ef_path):
                    try:
                        df_ = pd.read_csv(ef_path)
                        if 'EF_J_m' in df_.columns and len(df_) > 0:
                            series = df_['EF_J_m'].dropna()
                            if apcemm_style and len(series) > 1:
                                series = series.iloc[1:]
                            if len(series) > 0:
                                per_case[tid][m][tt].append(float(series.iloc[-1]))
                    except Exception:
                        pass

# compute stats (mean/min/max) per case
stats_case = {tid: {m: {tt: None for tt in time_tags} for m in models} for tid in test_ids}
for tid in test_ids:
    for m in models:
        for tt in time_tags:
            vals = np.array(per_case[tid][m][tt], dtype=float)
            if vals.size:
                stats_case[tid][m][tt] = {'mean': vals.mean(), 'min': vals.min(), 'max': vals.max()}

## Get CoCiP internal EF values

## Plot results

In [ ]:
# ── layout ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=300, sharey=False)

# ── group x-axis by temperature → RHi ────────────────────────────────────────
temps_unique = sorted({int(tid.split('T')[1].split('L')[0]) for tid in test_ids})

rhi_labels  = ['110%', '130%']
rh_to_label = {v: k for k, v in rh_color.items()}

groups = []
for T in temps_unique:
    tids_at_T = sorted(
        [tid for tid in test_ids if f"T{T}" in tid],
        key=lambda s: int(s[:3])
    )
    for tid in tids_at_T:
        groups.append((T, tid))

n_groups    = len(groups)
model_order = ['CoCiP', 'LES', 'APCEMM']
n_models    = len(model_order)
bar_width   = 0.22
group_gap   = 0.12

def group_x_positions(groups, temps_unique, bar_width, group_gap):
    x0, positions, prev_T = 0.0, [], None
    for (T, tid) in groups:
        if prev_T is not None and T != prev_T:
            x0 += group_gap
        positions.append(x0)
        x0 += n_models * bar_width + 0.10
        prev_T = T
    return positions

group_starts = group_x_positions(groups, temps_unique, bar_width, group_gap)

model_colors = {'CoCiP': 'tab:green', 'LES': 'tab:orange', 'APCEMM': 'tab:blue'}
model_colors_light = {
    'CoCiP':  '#a8d5a2',
    'LES':    '#ffd699',
    'APCEMM': '#a8c4e8',
}

x_left  = -0.2
x_right = group_starts[-1] + n_models * bar_width + 0.2

# ── helper: draw all bars + errorbars onto a given axes ──────────────────────
def draw_bars(ax, tt):
    for gi, (T, tid) in enumerate(groups):
        x_group = group_starts[gi]
        for mi, m in enumerate(model_order):
            xc        = x_group + mi * bar_width
            xc_center = xc + bar_width / 2
            col       = model_colors[m]
            col_light = model_colors_light[m]

            s4 = stats_case_4h.get(tid, {}).get(m, {}).get(tt, None)
            sf = stats_case.get(tid, {}).get(m, {}).get(tt, None)

            if sf is not None:
                yf    = sf['mean']
                ax.bar(xc, abs(yf), bottom=min(yf, 0), width=bar_width,
                       align='edge', color=col_light, edgecolor=col,
                       linewidth=0.8, hatch='', zorder=2)
                ax.errorbar(xc_center, yf,
                            yerr=[[yf - sf['min']], [sf['max'] - yf]],
                            fmt='o', color=col, ecolor=col, elinewidth=0.8,
                            capsize=2, markersize=3, zorder=4)

            if s4 is not None:
                y4    = s4['mean']
                ax.bar(xc, abs(y4), bottom=min(y4, 0), width=bar_width,
                       align='edge', color='None', edgecolor=col,
                       linewidth=0.8, hatch='///', zorder=2)
                ax.errorbar(xc_center, y4,
                            yerr=[[y4 - s4['min']], [s4['max'] - y4]],
                            fmt='o', color=col, ecolor=col, elinewidth=0.8,
                            capsize=2, markersize=3, zorder=4)

# ── CoCiP EF_internal extraction ─────────────────────────────────────────────
def get_cocip_internal_point(tid, tt, target_h):
    # no noon data yet
    if tt == '12h':
        return np.nan

    max_contrail_age = cocip_bypass_data[tid]['age_hours'].values[-1]
    if target_h is None:
        idx = -1
    elif target_h >= max_contrail_age:
        return np.nan
    else:
        idx = np.argmin(np.abs(cocip_bypass_data[tid]['age_hours'].values.astype(float) - target_h))

    ef_cocip_internal = np.cumsum(
        cocip_bypass_data[tid]["ef"].values.astype(float)[1:] /
        cocip_bypass_data[tid]["segment_length"].values.astype(float)[1:]
    )  # J -> J/m cumulative
    print(f"tid={tid}, tt={tt}, target_h={target_h} -> idx={idx}, age={cocip_bypass_data[tid]['age_hours'].values[idx]:.2f} h, ef_internal={ef_cocip_internal[idx]:.2e} J/m")
    return ef_cocip_internal[idx]

# key: (tid, tt, mode) where mode in {"none", "4h"}
cocip_internal = {}
for tid in test_ids:
    for tt in time_tags:
        cocip_internal[(tid, tt, 'none')] = get_cocip_internal_point(tid, tt, None)
        cocip_internal[(tid, tt, '4h')]   = get_cocip_internal_point(tid, tt, TARGET_AGE_H)

# ── main panels ───────────────────────────────────────────────────────────────
cocip_idx = model_order.index('CoCiP')
for ax, tt, title in zip(axes, time_tags, ['LW Only', 'LW + SW']):
    draw_bars(ax, tt)

    # overlay CoCiP EF_internal as circles at CoCiP bar center
    for gi, (_, tid) in enumerate(groups):
        x = group_starts[gi] + (cocip_idx + 0.5) * bar_width

        # None/full: closed circle
        y_none = cocip_internal.get((tid, tt, 'none'), np.nan)
        if np.isfinite(y_none):
            ax.scatter(x, y_none, marker='s', s=12, linewidths=1.0,
                       facecolors='black', edgecolors='black', zorder=6)

        # 4h: open circle
        y_4h = cocip_internal.get((tid, tt, '4h'), np.nan)
        if np.isfinite(y_4h):
            ax.scatter(x, y_4h, marker='s', s=12, linewidths=1.2,
                       facecolors='none', edgecolors='black', zorder=6)

    ax.axhline(0, color='gray', linestyle='--', lw=0.8)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xticks([])
    ax.set_xlim(x_left, x_right)

# ── y limits ──────────────────────────────────────────────────────────────────
ax_ylims = {}
for ax, tt in zip(axes, time_tags):
    ymins, ymaxs = [], []
    for tid in test_ids:
        for m in model_order:
            for s in [stats_case_4h.get(tid, {}).get(m, {}).get(tt),
                      stats_case.get(tid, {}).get(m, {}).get(tt)]:
                if s:
                    ymins.append(s['min'])
                    ymaxs.append(s['max'])

        # include both CoCiP EF_internal points in limits
        for mode in ['none', '4h']:
            y_int = cocip_internal.get((tid, tt, mode), np.nan)
            if np.isfinite(y_int):
                ymins.append(y_int)
                ymaxs.append(y_int)

    ymin = min(ymins) if ymins else -1
    ymax = max(ymaxs) if ymaxs else 1

    pad = 0.06 * (ymax - ymin)
    if tt == '0h':
        ax.set_ylim(ymin, ymax + pad)
    else:
        ax.set_ylim(ymin - pad, ymax + pad)

    ax.set_ylabel('EF [J m$^{-1}$]', fontsize=12)
    ax_ylims[tt] = ax.get_ylim()

# ── x-tick labels ─────────────────────────────────────────────────────────────
for ax, tt, title in zip(axes, time_tags, ['LW Only', 'LW + SW']):
    ymin_ax, ymax_ax = ax_ylims[tt]
    ybot = ymin_ax
    for gi, (T, tid) in enumerate(groups):
        xc = group_starts[gi] + (n_models * bar_width) / 2
        rhi_str = tid[:3] + '%'
        offset = 0.02e10 if title == 'LW Only' else 0.05e9
        ax.text(xc, ybot - offset, rhi_str, ha='center', va='top', fontsize=12,
                transform=ax.transData, clip_on=False)

    prev_T, block_start_x = None, None
    def flush_temp_label(ax, T, bsx, bex, ybot, ymn, ymx):
        xc = (bsx + bex) / 2
        ax.text(xc, ybot - 0.08 * (ymx - ymn),
                f'{T} K', ha='center', va='top', fontsize=12, fontweight='bold',
                transform=ax.transData, clip_on=False)

    for gi, (T, tid) in enumerate(groups):
        if T != prev_T:
            if prev_T is not None:
                flush_temp_label(ax, prev_T, block_start_x,
                                 group_starts[gi-1] + n_models * bar_width,
                                 ybot, ymin_ax, ymax_ax)
            block_start_x = group_starts[gi]
            prev_T = T
    flush_temp_label(ax, prev_T, block_start_x,
                     group_starts[-1] + n_models * bar_width,
                     ybot, ymin_ax, ymax_ax)

# ── legend along bottom of figure ────────────────────────────────────────────
model_handles = [
    mpatches.Patch(facecolor=model_colors_light[m], edgecolor=model_colors[m],
                   linewidth=0.8, label=m)
    for m in model_order
]
full_patch  = mpatches.Patch(facecolor='lightgray', edgecolor='gray',
                             linewidth=0.8, label='full run')
trunc_patch = mpatches.Patch(facecolor='None', edgecolor='gray',
                             hatch='///', label='4h truncated')

internal_none_handle = plt.Line2D(
    [0], [0], marker='o', linestyle='None', color='black',
    markerfacecolor='black', markeredgecolor='black',
    markersize=6, label='CoCiP Internal'
)
internal_4h_handle = plt.Line2D(
    [0], [0], marker='o', linestyle='None', color='black',
    markerfacecolor='none', markeredgecolor='black',
    markersize=6, label='CoCiP Internal (4h)'
)

fig.legend(
    handles=model_handles + [full_patch, trunc_patch, internal_none_handle, internal_4h_handle],
    loc='lower center', ncol=8, fontsize=10, framealpha=0.9,
    bbox_to_anchor=(0.5, -0.06)
)

plt.tight_layout()
plt.subplots_adjust(bottom=0.14)
plt.show()

In [ ]:
# CoCiP-only EF plot (external + internal), styled like Figure 3

# ---- grouping (Temperature -> RH_i) ----
temps_unique = sorted({int(tid.split("T")[1].split("L")[0]) for tid in test_ids})
groups = []
for T in temps_unique:
    tids_at_T = sorted([tid for tid in test_ids if f"T{T}" in tid], key=lambda s: int(s[:3]))
    for tid in tids_at_T:
        groups.append((T, tid))

# x positions with small gaps between temperature blocks
group_starts = []
x0 = 0.0
prev_T = None
for T, _ in groups:
    if prev_T is not None and T != prev_T:
        x0 += 0.20
    group_starts.append(x0)
    x0 += 0.45
    prev_T = T

# ---- CoCiP internal EF from bypass ----
def cocip_internal_from_bypass(tid, tt, target_h=None):
    if tt == "12h":  # no noon/internal stream available in current setup
        return np.nan

    ds = cocip_bypass_data[tid]
    age = ds["age_hours"].values.astype(float)
    ef = ds["ef"].values.astype(float)
    seg = ds["segment_length"].values.astype(float)

    # cumulative J -> J/m, skipping first element (same convention as above)
    ef_cum = np.cumsum(ef[1:] / seg[1:])
    age_cum = age[1:]
    if ef_cum.size == 0 or age_cum.size == 0:
        return np.nan

    if target_h is None:
        return float(ef_cum[-1])

    if target_h >= np.nanmax(age_cum):
        return np.nan

    idx = int(np.nanargmin(np.abs(age_cum - target_h)))
    return float(ef_cum[idx])

cocip_internal = {}
for tid in test_ids:
    for tt in time_tags:
        cocip_internal[(tid, tt, "full")] = cocip_internal_from_bypass(tid, tt, None)
        cocip_internal[(tid, tt, "4h")] = cocip_internal_from_bypass(tid, tt, TARGET_AGE_H)

# ---- plot ----
fig, axes = plt.subplots(1, 2, figsize=(11, 5), dpi=300, sharey=False)

bar_color = "#a8d5a2"
edge_color = "tab:green"

for ax, tt, title in zip(axes, time_tags, ["LW Only", "LW + SW"]):
    y_all = []

    for gi, (T, tid) in enumerate(groups):
        x = group_starts[gi]
        sf = stats_case.get(tid, {}).get("CoCiP", {}).get(tt, None)
        s4 = stats_case_4h.get(tid, {}).get("CoCiP", {}).get(tt, None)

        # full run (solid light bar)
        if sf is not None:
            yf = sf["mean"]
            ax.bar(
                x, abs(yf), bottom=min(yf, 0), width=0.30,
                align="center", color=bar_color, edgecolor=edge_color, linewidth=0.8, zorder=2
            )
            ax.errorbar(
                x, yf,
                yerr=[[yf - sf["min"]], [sf["max"] - yf]],
                fmt="o", color=edge_color, ecolor=edge_color, elinewidth=0.8,
                capsize=2, markersize=3, zorder=4
            )
            y_all += [sf["min"], sf["max"]]

        # 4h truncated (hatched overlay)
        if s4 is not None:
            y4 = s4["mean"]
            ax.bar(
                x, abs(y4), bottom=min(y4, 0), width=0.30,
                align="center", color="none", edgecolor=edge_color, linewidth=0.8, hatch="///", zorder=3
            )
            ax.errorbar(
                x, y4,
                yerr=[[y4 - s4["min"]], [s4["max"] - y4]],
                fmt="o", color=edge_color, ecolor=edge_color, elinewidth=0.8,
                capsize=2, markersize=3, zorder=5
            )
            y_all += [s4["min"], s4["max"]]

        # internal points: filled square (full), open square (4h)
        y_int_full = cocip_internal.get((tid, tt, "full"), np.nan)
        y_int_4h = cocip_internal.get((tid, tt, "4h"), np.nan)

        if np.isfinite(y_int_full):
            ax.scatter(x, y_int_full, marker="s", s=16, facecolors="black", edgecolors="black", zorder=6)
            y_all.append(y_int_full)
        if np.isfinite(y_int_4h):
            ax.scatter(x, y_int_4h, marker="s", s=16, facecolors="none", edgecolors="black", linewidths=1.1, zorder=6)
            y_all.append(y_int_4h)

    ax.axhline(0, color="gray", linestyle="--", lw=0.8)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel("EF [J m$^{-1}$]")

    # RH labels as ticks
    ax.set_xticks(group_starts)
    ax.set_xticklabels([f"{tid[:3]}%" for _, tid in groups], fontsize=10)

    # temperature labels under each block
    ymin, ymax = ax.get_ylim()
    span = ymax - ymin if ymax > ymin else 1.0
    for T in temps_unique:
        idxs = [i for i, (TT, _) in enumerate(groups) if TT == T]
        if not idxs:
            continue
        x_mid = 0.5 * (group_starts[idxs[0]] + group_starts[idxs[-1]])
        ax.text(x_mid, ymin - 0.09 * span, f"{T} K", ha="center", va="top",
                fontsize=10, fontweight="bold", clip_on=False)

# legend
legend_handles = [
    mpatches.Patch(facecolor=bar_color, edgecolor=edge_color, linewidth=0.8, label="CoCiP (full run)"),
    mpatches.Patch(facecolor="none", edgecolor=edge_color, hatch="///", linewidth=0.8, label="CoCiP (4h truncated)"),
    Line2D([0], [0], marker="s", linestyle="None", color="black", markerfacecolor="black", markersize=5, label="EF_internal (full)"),
    Line2D([0], [0], marker="s", linestyle="None", color="black", markerfacecolor="none", markersize=5, label="EF_internal (4h)"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.subplots_adjust(bottom=0.20)
plt.show()

In [ ]:
# EF tables from Figure 3 stats, formatted as: mean ± uncertainty_range
# uncertainty_range here is half-width of the plotted min/max range: (max - min)/2

model_cols = ['CoCiP', 'LES', 'APCEMM'] if 'model_order' in globals() else list(models.keys())

def case_label_from_tid(tid):
    return f"{tid[:3]} {tid.split('T')[1].split('L')[0]}K"

def fmt_mean_pm_range(stat):
    if not stat or any(k not in stat for k in ("mean", "min", "max")):
        return "nan"
    mean = float(stat["mean"])
    unc = 0.5 * (float(stat["max"]) - float(stat["min"]))
    return f"{mean:.2e} ± {unc:.2e}"

# Full-run mean ± range
ef_full_by_tt = {}
for tt in time_tags:
    ef_full_by_tt[tt] = pd.DataFrame(
        {
            m: [fmt_mean_pm_range(stats_case.get(tid, {}).get(m, {}).get(tt, None)) for tid in test_ids]
            for m in model_cols
        },
        index=[case_label_from_tid(tid) for tid in test_ids]
    )

ef_full_table = pd.concat(ef_full_by_tt, axis=1)
ef_full_table.index.name = "Case"
ef_full_table.columns.names = ["Time Tag", "Model"]

# 4h-truncated mean ± range
ef_4h_by_tt = {}
for tt in time_tags:
    ef_4h_by_tt[tt] = pd.DataFrame(
        {
            m: [fmt_mean_pm_range(stats_case_4h.get(tid, {}).get(m, {}).get(tt, None)) for tid in test_ids]
            for m in model_cols
        },
        index=[case_label_from_tid(tid) for tid in test_ids]
    )

ef_4h_table = pd.concat(ef_4h_by_tt, axis=1)
ef_4h_table.index.name = "Case"
ef_4h_table.columns.names = ["Time Tag", "Model"]

print("Full-run EF means (formatted):")
display(ef_full_table)

print("4h-truncated EF means (formatted):")
display(ef_4h_table)

# Figure 4: ANOVA EF Bar Chart